In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from PI_NN_load_call import PE_call as PE, should_use_compensated_stopping
from PolicyIteration_call import Policy_Iteration as PI, make_experiment_tag
from call_benchmarks import benchmark_gbm_geometric_call_fdm, compute_1d_call_reference_values


### Set $d$ = 1 for Example 1 call robustness, $d$ = 10 for Example 2 high-dimensional geometric call robustness.

In [ ]:
d = 1

total_time = 0.5
n_time_steps = 100
dt = total_time / n_time_steps
K = 10.0
r = 0.05
strike = 40.0
x_0 = 40.0
dividend_train = 0.05
dividend_true_list = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25]

lambda_temp = 1
use_compensated_stopping = should_use_compensated_stopping(lambda_temp)

if d == 1:
    sigma = 0.4
    hidden_dim = 21
    test_size = 2**20
    epsilon_candidates = [0.0, 0.1, 0.2, 0.3]
    model_dir = "call_trained_models_robust_PI"
    results_path = f"call_robustness_results_lambda_{lambda_temp:g}_penalty_{K}.csv"
    raw_results_path = None
    fdm_reference_path = None
    experiment_tag = None
else:
    sigma = 0.2
    hidden_dim = 20 + d
    test_size = 2**18
    epsilon_candidates = [0.0, 0.2, 0.4]
    experiment_tag = make_experiment_tag(d=d, sigma=sigma, dividend=dividend_train, x_0=x_0)
    model_dir = (
        "call_trained_models_robust_PI_highdim_gbm-2"
        if use_compensated_stopping
        else "call_trained_models_robust_PI_highdim_gbm"
    )
    stopping_rule_suffix = "_compensated" if use_compensated_stopping else ""
    raw_results_path = f"call_highdim_gbm_robustness_raw_{experiment_tag}_lambda_{lambda_temp}_K_{K}{stopping_rule_suffix}.csv"
    fdm_reference_path = f"call_highdim_gbm_robustness_fdm_reference_{experiment_tag}.csv"
    results_path = f"call_highdim_gbm_robustness_{experiment_tag}_lambda_{lambda_temp}_K_{K}{stopping_rule_suffix}.csv"

torch.set_default_dtype(torch.float64)
device = torch.device("cpu")

print(f"d={d}, lambda={lambda_temp}, compensated={use_compensated_stopping}")
print(f"model_dir={model_dir}")


In [ ]:
train_models = False if d == 1 else True
train_batch_size = 2**10
PE_iteration = 1000 if d == 1 else 1500
# PI_iteration = 10
PI_iteration = 3
trained_solver_lookup = {}

if train_models:
    solver_list = [
        PI(
            d=d,
            total_time=total_time,
            n_time_steps=n_time_steps,
            K=K,
            r=r,
            dividend=dividend_train,
            sigma=sigma,
            strike=strike,
            x_0=x_0,
            lambda_temp=lambda_temp,
            epsilon=epsilon,
            device=device,
            hidden_layers=2,
            hidden_dim=hidden_dim,
            lr=0.001,
            model_save_flag=1,
            model_dir=model_dir,
            model_tag=experiment_tag,
        )
        for epsilon in epsilon_candidates
    ]

    for solver in solver_list:
        solver.PolicyIteration(
            PI_iteration=PI_iteration,
            PE_iteration=PE_iteration,
            batch_size=train_batch_size,
        )

    trained_solver_lookup = {solver.epsilon: solver for solver in solver_list}
    print(f"Finished training {len(trained_solver_lookup)} models.")
else:
    print("Skipping training.")


In [ ]:
def simulate_forward_process_misspecification(batch_size, dividend_true):
    x = torch.zeros(batch_size, n_time_steps + 1, d, device=device)
    x[:, 0, :] = x_0
    for t in range(n_time_steps):
        dw = np.sqrt(dt) * torch.randn(batch_size, d, device=device)
        x[:, t + 1, :] = x[:, t, :] * torch.exp(
            (r - dividend_true - 0.5 * sigma**2) * dt + sigma * dw
        )
    return x


def compute_reference_df():
    if d == 1:
        return pd.DataFrame(
            compute_1d_call_reference_values(
                dividend_list=dividend_true_list,
                x_0=x_0,
                strike=strike,
                r=r,
                sigma=sigma,
                total_time=total_time,
                Nt=1000,
                Ny=4000,
            )
        )

    reference_rows = []
    for dividend_true in dividend_true_list:
        fdm_value, reduced_params = benchmark_gbm_geometric_call_fdm(
            d=d,
            x_0=x_0,
            strike=strike,
            r=r,
            dividend=dividend_true,
            sigma=sigma,
            total_time=total_time,
            Nt=1500,
            Ny=5000,
            penalty_K=1e8,
            y_range=3.5,
        )
        reference_rows.append(
            {
                "dividend_true": dividend_true,
                "reference_fdm_1d": fdm_value,
                "x0_eff": reduced_params["x0_eff"],
                "sigma_eff": reduced_params["sigma_eff"],
                "dividend_eff": reduced_params["dividend_eff"],
            }
        )

    reference_df = pd.DataFrame(reference_rows)
    reference_df.to_csv(fdm_reference_path, index=False)
    print(f"Saved reduced 1D FDM reference: {fdm_reference_path}")
    return reference_df


def find_trained_models():
    model_files = list(Path(model_dir).glob("*.pt"))
    models = []

    if d == 1:
        for model_file in model_files:
            parts = model_file.stem.split("_")
            try:
                epsilon = float(parts[2])
                lambda_value = float(parts[4])
            except (IndexError, ValueError):
                continue
            models.append(
                {
                    "model_name": model_file.stem,
                    "epsilon": epsilon,
                    "lambda_temp": lambda_value,
                }
            )
        return sorted(models, key=lambda item: (item["lambda_temp"], item["epsilon"]))

    for model_file in model_files:
        model_name = model_file.stem
        prefix = f"{experiment_tag}_eps_"
        if not model_name.startswith(prefix):
            continue

        remainder = model_name[len(prefix):]
        parts = remainder.split("_lamda_")
        if len(parts) != 2:
            continue

        epsilon_str, remainder = parts
        parts = remainder.split("_penalty_")
        if len(parts) != 2:
            continue

        lambda_str, penalty_str = parts

        try:
            epsilon = float(epsilon_str)
            lambda_value = float(lambda_str)
            penalty = float(penalty_str)
        except ValueError:
            continue

        models.append(
            {
                "model_name": model_name,
                "epsilon": epsilon,
                "lambda_temp": lambda_value,
                "penalty": penalty,
            }
        )
    return sorted(models, key=lambda item: (item["lambda_temp"], item["epsilon"]))


reference_df = compute_reference_df()

if train_models:
    trained_models = [
        {
            "model_name": solver._model_name(),
            "epsilon": solver.epsilon,
            "lambda_temp": solver.lambda_temp,
        }
        for solver in solver_list
    ]
    print(f"Using {len(trained_models)} trained models just now.")
else:
    trained_models = find_trained_models()
    print(f"Found {len(trained_models)} matched models.")

trained_models


In [ ]:
eval_models = [
    model
    for model in trained_models
    if abs(model["lambda_temp"] - lambda_temp) < 1e-12
    and any(abs(model["epsilon"] - eps) < 1e-12 for eps in epsilon_candidates)
]
if not eval_models:
    raise RuntimeError("No matched trained models were found for the chosen lambda_temp.")

print(f"Starting evaluation of {len(eval_models)} models...")
start_time = time.time()
results = []
raw_results = []

for dividend_true in dividend_true_list:
    x_test = simulate_forward_process_misspecification(test_size, dividend_true)

    if d == 1:
        ref_row = reference_df[reference_df["dividend_true"].apply(lambda x: abs(x - dividend_true) < 1e-12)]
        reference_value = ref_row.iloc[0]["reference_value"]

    for model in eval_models:
        epsilon = model["epsilon"]
        eval_solver = PE(
            d=d,
            total_time=total_time,
            n_time_steps=n_time_steps,
            K=K,
            r=r,
            dividend=dividend_true,
            sigma=sigma,
            strike=strike,
            x_0=x_0,
            lambda_temp=lambda_temp,
            epsilon=epsilon,
            use_compensated_stopping=use_compensated_stopping,
            hidden_layers=2,
            hidden_dim=hidden_dim,
            test_size=test_size,
            device=device,
        )

        if train_models:
            trained_solver = trained_solver_lookup.get(epsilon)
            if trained_solver is None:
                continue
            eval_solver.load_state_dict(trained_solver.policy_evaluation_NN_solver.state_dict())
            model_loaded = True
        else:
            model_loaded = eval_solver.load_model(model_dir, model["model_name"])

        if model_loaded:
            eval_start = time.time()
            expected_reward, er_std = eval_solver.evaluate_expected_reward(x_test)
            evaluation_time = time.time() - eval_start

            if d == 1:
                relative_error = abs(expected_reward - reference_value) / reference_value
                results.append(
                    {
                        "epsilon": epsilon,
                        "dividend_true": dividend_true,
                        "reference_value": reference_value,
                        "expected_reward": expected_reward,
                        "er_std": er_std,
                        "relative_error": relative_error,
                        "evaluation_time": evaluation_time,
                    }
                )
                print(
                    f"epsilon={epsilon}, dividend_true={dividend_true:.2f}, expected_reward={expected_reward:.6f}, RE={relative_error:.6f}"
                )
            else:
                raw_results.append(
                    {
                        "epsilon": epsilon,
                        "model_name": model["model_name"],
                        "dividend_true": dividend_true,
                        "expected_reward": expected_reward,
                        "er_std": er_std,
                        "evaluation_time": evaluation_time,
                    }
                )
                print(
                    f"epsilon={epsilon}, dividend_true={dividend_true:.2f}, expected_reward={expected_reward:.6f}"
                )

if d == 1:
    results_df = pd.DataFrame(results).sort_values(["epsilon", "dividend_true"]).reset_index(drop=True)
    results_df.to_csv(results_path, index=False)
    print(f"Saved robustness results: {results_path}")
else:
    raw_results_df = pd.DataFrame(raw_results).sort_values(["epsilon", "dividend_true"]).reset_index(drop=True)
    raw_results_df.to_csv(raw_results_path, index=False)
    print(f"Saved raw evaluation results: {raw_results_path}")

    results_df = raw_results_df.merge(
        reference_df[["dividend_true", "reference_fdm_1d"]],
        on="dividend_true",
        how="left",
    )
    results_df["relative_error_fdm_1d"] = (
        (results_df["expected_reward"] - results_df["reference_fdm_1d"]).abs()
        / results_df["reference_fdm_1d"].abs()
    )
    results_df.to_csv(results_path, index=False)
    print(f"Saved robustness results: {results_path}")

print(f"Evaluation completed in {time.time() - start_time:.2f} seconds.")
results_df.head()


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
epsilon_colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#27d6b6"]


def add_training_dividend_line(ax):
    ax.axvline(
        x=dividend_train,
        color="red",
        linestyle="--",
        linewidth=3 if d == 1 else 2.5,
        alpha=0.6,
        zorder=10,
        label=f"Training Dividend Rate ({dividend_train})" if d == 1 else f"Training dividend {dividend_train}",
    )


if d == 1:
    output_dir = "call_robustness_performance"
    os.makedirs(output_dir, exist_ok=True)

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    for i, eps in enumerate(sorted(results_df["epsilon"].unique())):
        eps_data = results_df[results_df["epsilon"] == eps].sort_values("dividend_true")
        ax.plot(
            eps_data["dividend_true"],
            eps_data["relative_error"] * 100,
            "o-",
            color=epsilon_colors[i % len(epsilon_colors)],
            linewidth=5,
            markersize=8,
            label=f"epsilon={eps}",
            alpha=0.6,
        )

    add_training_dividend_line(ax)
    ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.6, linewidth=3, label="1% reference line")
    ax.set_xlabel("True Dividend Rate", fontsize=37)
    ax.set_ylabel("Relative Error (%)", fontsize=37)
    ax.set_title(f"lambda = {lambda_temp:g}", fontsize=37)
    ax.legend(loc="upper right" if use_compensated_stopping else "best", fontsize=24)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", labelsize=35)
    ax.tick_params(axis="y", labelsize=35)
    if use_compensated_stopping:
        ax.set_ylim(0, 10)

    plt.tight_layout()
    filename = os.path.join(
        output_dir,
        f"call_RE_lambda_{lambda_temp:g}{'-compensated' if use_compensated_stopping else ''}.pdf",
    )
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    print(f"Saved: {filename}")
    plt.show()
    plt.close()
else:
    output_dir = "call_highdim_robustness_performance"
    os.makedirs(output_dir, exist_ok=True)

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    for i, eps in enumerate(sorted(results_df["epsilon"].unique())):
        eps_data = results_df[results_df["epsilon"] == eps].sort_values("dividend_true")
        ax.plot(
            eps_data["dividend_true"],
            eps_data["relative_error_fdm_1d"] * 100,
            "o-",
            linewidth=4,
            label=f"$\epsilon$={eps}",
            color=epsilon_colors[i % len(epsilon_colors)],
        )

    add_training_dividend_line(ax)
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=2.0, alpha=0.6, label="1% reference line")
    ax.set_xlabel("True Dividend Rate", fontsize=37)
    ax.set_ylabel("Relative Error (%)", fontsize=37)
    ax.set_title(f"d={d}, $\lambda={lambda_temp}$, geometric call", fontsize=37)
    ax.legend(loc="best", fontsize=24)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", labelsize=35)
    ax.tick_params(axis="y", labelsize=35)

    plt.tight_layout()
    filename = (
        f"{output_dir}/gbm_{experiment_tag}_lambda_{lambda_temp}_vs_fdm_1d"
        f"{'_compensated' if use_compensated_stopping else ''}.pdf"
    )
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    print(f"Saved: {filename}")
    plt.show()
    plt.close()
